# PEA Electricity Demand Data – Unpivot & Data Preparation

Notebook สำหรับรวมไฟล์ .xls จากโฟลเดอร์ data และแปลงข้อมูลจาก Wide Format เป็น Long Format เพื่อเตรียมใช้กับ Machine Learning

กติกาการประมวลผล
- อ่านชีตชื่อ Source โดยใช้ Pandas
- ใช้แถวที่ 4 เป็นชื่อ customer_type
- อ่าน day_type จากแถวหัวข้อกลุ่ม เช่น PEAK DAY, WORKDAY แล้วกำหนดให้กับข้อมูลด้านล่างจนกว่าจะพบหัวข้อกลุ่มถัดไป
- ตัด record เวลา 00:00 ออก
- แปลงข้อมูลทุก customer type เป็นแถว
- อ่าน region, year และ month จากชื่อไฟล์ เช่น sg14170100.xls
> ก่อนรันครั้งแรก ให้ติดตั้ง xlrd เพื่อให้ Pandas อ่านไฟล์ Excel .xls ได้

## 1. Import libraries และกำหนดค่าเริ่มต้น

In [50]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from datetime import datetime, time

In [58]:
# โฟลเดอร์เก็บไฟล์ต้นฉบับ .xls
DATA_DIR = Path("data")

# ชื่อชีตในไฟล์ Excel
SHEET_NAME = "Source"

# แถวของ Header (นับแบบ Excel เริ่มจาก 1)
CUSTOMER_HEADER_EXCEL_ROW = 4

# Mapping รหัสภูมิภาค: แก้ไข/เพิ่มรหัสได้ตามข้อมูลจริง
REGION_MAP = {
    "14": "North",
    "15": "Region_15",  # TODO: ระบุชื่อภูมิภาคที่ถูกต้อง
}

# Mapping เดือน (ใช้เลขเดือนเป็นหลัก; เพิ่มชื่อเดือนได้หากต้องการ)
MONTH_MAP = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December",
}

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [59]:
FILENAME_PATTERN = re.compile(
    r"^sg(?P<region>\d{2})(?P<year>\d{2})(?P<month>\d{2})(?P<suffix>\d*)$",
    flags=re.IGNORECASE,
)

def get_file_metadata(file_path: Path) -> dict:
    """แยก region, year, month จากชื่อไฟล์ เช่น sg14170100.xls"""
    match = FILENAME_PATTERN.match(file_path.stem)
    if not match:
        raise ValueError(
            f"ชื่อไฟล์ไม่ตรงรูปแบบที่กำหนด: {file_path.name} "
            "(ตัวอย่างที่รองรับ: sg14170100.xls)"
        )

    region_code = match.group("region")
    year = 2000 + int(match.group("year"))
    month = int(match.group("month"))

    if not 1 <= month <= 12:
        raise ValueError(f"เลขเดือนไม่ถูกต้องในชื่อไฟล์: {file_path.name}")

    return {
        "region_code": region_code,
        "region": REGION_MAP.get(region_code, f"Unknown_{region_code}"),
        "year": year,
        "month": month,
        "month_name": MONTH_MAP[month],
    }

# ทดลองแยก metadata จากไฟล์ตัวอย่าง
get_file_metadata(Path("sg14170100.xls"))


{'region_code': '14',
 'region': 'North',
 'year': 2017,
 'month': 1,
 'month_name': 'January'}

In [62]:
def parse_time_value(value):
    """แปลงค่าเวลาใน Excel ให้เป็นข้อความ HH:MM; คืน None หากไม่ใช่เวลา"""
    if pd.isna(value):
        return None

    if isinstance(value, (datetime, time, pd.Timestamp)):
        return value.strftime("%H:%M")

    # Excel อาจเก็บเวลาเป็นเศษส่วนของ 1 วัน เช่น 0.0104167 = 00:15
    if isinstance(value, (int, float, np.integer, np.floating)):
        numeric_value = float(value)
        if 0 <= numeric_value < 1:
            total_minutes = int(round(numeric_value * 24 * 60)) % (24 * 60)
            return f"{total_minutes // 60:02d}:{total_minutes % 60:02d}"
        return None

    text_value = str(value).strip()
    match = re.search(r"(?<!\d)(\d{1,2}):(\d{2})(?!\d)", text_value)
    
    if match:
        hour, minute = int(match.group(1)), int(match.group(2))
        if 0 <= hour <= 24 and 0 <= minute <= 59:
            return f"{hour:02d}:{minute:02d}"
    return None


In [63]:
def clean_one_file(file_path: Path) -> pd.DataFrame:
    """อ่านและแปลงไฟล์ Excel หนึ่งไฟล์เป็น Long Format"""
    metadata = get_file_metadata(file_path)

    # อ่านแบบไม่มี header เพื่อรักษาแถวหัวข้อ day_type ที่แทรกอยู่ในข้อมูล
    raw = pd.read_excel(
        file_path,
        sheet_name=SHEET_NAME,
        header=None,
        engine="calamine",
    )

    header_idx = CUSTOMER_HEADER_EXCEL_ROW - 1
    if len(raw) <= header_idx:
        raise ValueError(f"ไม่พบแถว Header แถวที่ {CUSTOMER_HEADER_EXCEL_ROW}: {file_path.name}")

    # คอลัมน์แรกคือ TIME; คอลัมน์ที่เหลือในแถว 4 คือ customer type
    header_row = raw.iloc[header_idx]
    customer_columns = {}
    for col_idx in range(1, len(raw.columns)):
        header_value = header_row.iloc[col_idx]
        if pd.notna(header_value) and str(header_value).strip():
            customer_columns[col_idx] = str(header_value).strip()

    if not customer_columns:
        raise ValueError(f"ไม่พบ customer_type ในแถว {CUSTOMER_HEADER_EXCEL_ROW}: {file_path.name}")

    records = []
    current_day_type = None

    # เริ่มอ่านหลังแถว Header ของ customer_type
    for row_idx in range(header_idx + 1, len(raw)):
        row = raw.iloc[row_idx]
        parsed_time = parse_time_value(row.iloc[0])

        if parsed_time is None:
            # แถวที่ไม่มีเวลาและมีข้อความ ใช้เป็นตัวบอก day_type
            labels = []
            for value in row.tolist():
                if pd.notna(value) and isinstance(value, str) and value.strip():
                    labels.append(value.strip())
            if labels:
                # เลือกข้อความแรกที่ไม่ใช่ TIME และไม่ใช่ชื่อ customer type
                candidates = [
                    label for label in labels
                    if label.upper() != "TIME" and label not in customer_columns.values()
                ]
                if candidates:
                    current_day_type = candidates[0]
            continue

        # ตาม requirement ให้ตัดเวลา 00:00 ออก
        if parsed_time == "00:00":
            continue

        if current_day_type is None:
            raise ValueError(
                f"พบแถวข้อมูลเวลา {parsed_time} ก่อนพบ day_type ในไฟล์ {file_path.name} "
                f"(Excel row {row_idx + 1})"
            )

        for col_idx, customer_type in customer_columns.items():
            if col_idx >= len(row):
                continue
            demand_value = row.iloc[col_idx]
            if pd.isna(demand_value):
                continue
            if isinstance(demand_value, str):
                demand_value = demand_value.replace(",", "").strip()
            demand_value = pd.to_numeric(demand_value, errors="coerce")
            if pd.isna(demand_value):
                continue

            records.append({
                **metadata,
                "day_type": current_day_type,
                "time": parsed_time,
                "customer_type": customer_type,
                "demand_kw": float(demand_value),
                "source_file": file_path.name,
            })

    result = pd.DataFrame(records)
    if result.empty:
        raise ValueError(f"ไม่พบข้อมูล demand ที่แปลงได้ในไฟล์: {file_path.name}")

    return result

In [55]:
xls_files = sorted(DATA_DIR.glob("*.xls"))
if not xls_files:
    raise FileNotFoundError(f"ไม่พบไฟล์ .xls ในโฟลเดอร์: {DATA_DIR.resolve()}")

all_data = []
failed_files = []

for file_path in xls_files:
    try:
        file_df = clean_one_file(file_path)
        all_data.append(file_df)
        print(f"SUCCESS: {file_path.name} -> {len(file_df):,} rows")
    except Exception as error:
        failed_files.append({"file": file_path.name, "error": str(error)})
        print(f"FAILED : {file_path.name} -> {error}")

if not all_data:
    raise RuntimeError("ไม่มีไฟล์ใดประมวลผลสำเร็จ กรุณาตรวจสอบ error ด้านบน")

df = pd.concat(all_data, ignore_index=True)

print("Total files:", len(xls_files))
print("Successful files:", len(all_data))
print("Failed files:", len(failed_files))
print("Total records:", f"{len(df):,}")

if failed_files:
    display(pd.DataFrame(failed_files))


SUCCESS: sg14170100.xls -> 3,072 rows
Total files: 1
Successful files: 1
Failed files: 0
Total records: 3,072


In [56]:
# กำหนดลำดับคอลัมน์ให้อ่านง่าย
column_order = [
    "region_code", "region", "year", "month", "month_name",
    "day_type", "time", "customer_type", "demand_kw", "source_file"
]
df = df[column_order].copy()

# ตรวจสอบว่าไม่มีเวลา 00:00 เหลืออยู่
assert not df["time"].eq("00:00").any(), "พบ record เวลา 00:00 ที่ยังไม่ถูกลบ"

# แสดงตัวอย่างข้อมูล
display(df.head(10))
display(df.info())

# สรุปจำนวน record แยกตามไฟล์และ day_type
display(
    df.groupby(["source_file", "day_type"], dropna=False)
      .size()
      .rename("record_count")
      .reset_index()
)


,region_code,region,year,month,month_name,day_type,time,customer_type,demand_kw,source_file
0,14,North,2017,1,January,PEAK DAY,00:15,บ้านอยู่อาศัย < 150 หน่วย/เดือน,220452.89,sg14170100.xls
1,14,North,2017,1,January,PEAK DAY,00:15,บ้านอยู่อาศัย > 150 หน่วย/เดือน,292628.10,sg14170100.xls
2,14,North,2017,1,January,PEAK DAY,00:15,กิจการขนาดเล็ก,263279.80,sg14170100.xls
3,14,North,2017,1,January,PEAK DAY,00:15,กิจการขนาดกลาง,281908.01,sg14170100.xls
4,14,North,2017,1,January,PEAK DAY,00:15,กิจการขนาดใหญ่,610847.04,sg14170100.xls
5,14,North,2017,1,January,PEAK DAY,00:15,กิจการเฉพาะอย่าง,48087.54,sg14170100.xls
6,14,North,2017,1,January,PEAK DAY,00:15,องค์กรไม่แสวงหากำไร,29014.80,sg14170100.xls
7,14,North,2017,1,January,PEAK DAY,00:15,สูบน้ำเพื่อการเกษตร,13995.67,sg14170100.xls
8,14,North,2017,1,January,PEAK DAY,00:30,บ้านอยู่อาศัย < 150 หน่วย/เดือน,156171.69,sg14170100.xls
9,14,North,2017,1,January,PEAK DAY,00:30,บ้านอยู่อาศัย > 150 หน่วย/เดือน,316302.04,sg14170100.xls


<class 'pandas.DataFrame'>
RangeIndex: 3072 entries, 0 to 3071
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   region_code    3072 non-null   str    
 1   region         3072 non-null   str    
 2   year           3072 non-null   int64  
 3   month          3072 non-null   int64  
 4   month_name     3072 non-null   str    
 5   day_type       3072 non-null   str    
 6   time           3072 non-null   str    
 7   customer_type  3072 non-null   str    
 8   demand_kw      3072 non-null   float64
 9   source_file    3072 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 240.1 KB


None

,source_file,day_type,record_count
0,sg14170100.xls,PEAK DAY,768
1,sg14170100.xls,SATURDAY,768
2,sg14170100.xls,SUNDAY,768
3,sg14170100.xls,WORKDAY,768


In [57]:
csv_path = OUTPUT_DIR / "pea_demand_long_format.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Saved CSV: {csv_path.resolve()}")

try:
    parquet_path = OUTPUT_DIR / "pea_demand_long_format.parquet"
    df.to_parquet(parquet_path, index=False)
    print(f"Saved Parquet: {parquet_path.resolve()}")
except ImportError:
    print("ยังไม่ได้ติดตั้ง pyarrow จึงข้ามการบันทึก Parquet (ติดตั้งด้วย %pip install pyarrow ได้)")


Saved CSV: C:\Users\bang_\Desktop\PEA_Project\output\pea_demand_long_format.csv
Saved Parquet: C:\Users\bang_\Desktop\PEA_Project\output\pea_demand_long_format.parquet
